# Mistério em João Pessoa

Adaptação do lendário [SQL Murder Mystery](https://github.com/NUKnightLab/sql-mysteries) (Knight Lab / Northwestern University, conteúdo original sob licença CC BY-SA 4.0). Nesta adaptação você resolve tudo com pandas + MinIO. A cidade e alguns nomes/ruas viraram locais de João Pessoa.

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar.

## O que você recebeu

| Arquivo | Colunas | O que é |
|---|---|---|
| `ocorrencia.csv` | `data, tipo, descricao, cidade` | Boletins de Ocorrência |
| `pessoa.csv` | `id, nome, detran_id, numero_endereco, rua, cpf` | Cadastro de Pessoas |
| `detran.csv` | `id, idade, altura, cor_olhos, cor_cabelo, genero, placa, marca_veiculo, modelo_veiculo` | Cadastro do DETRAN |
| `depoimento.csv` | `pessoa_id, relato` | Depoimentos |
| `membro_academia.csv` | `id, pessoa_id, nome, data_matricula, plano` | Matrículas da Academia |
| `checkin_academia.csv` | `matricula_id, data_checkin, hora_entrada, hora_saida` | Check-ins da Academia |

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

In [1]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Conexão com o MinIO (S3-compatível) — pandas usa isso direto via s3fs
BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet** na camada — `bronze/<nome_tabela>.parquet`, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [2]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [3]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


In [4]:
# TODO: repita o padrão para "pessoa.csv" -> bronze.pessoa
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)

In [5]:
# TODO: repita o padrão para "detran.csv" -> bronze.detran
df_detran = pd.read_csv("dados/detran.csv")
df_detran.to_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS, index=False)

In [6]:
# TODO: repita o padrão para "depoimento.csv" -> bronze.depoimento
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)

In [7]:
# TODO: repita o padrão para "membro_academia.csv" -> bronze.membro_academia
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)

In [8]:
# TODO: repita o padrão para "checkin_academia.csv" -> bronze.checkin_academia
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)

Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de agora é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Pesquise a ocorrência e bus as testemunhas em `pessoa`.
2. Pesquise o `depoimento` das respectivas testemunhas.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` a outra para `detran`.
4. Pesquise pela pistas, mencionadas pelas testemunhas, nas tabelas correspondentes.
5. Confirme em `checkin_academia` que o suspeito tem um check-in na academia na data que a segunda testemunha mencionou.

In [9]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

### Passo 1 — a ocorrência

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**.

In [ ]:
# TODO:
caso = ocorrencia[
    (ocorrencia["data"] == 20180115) &
    (ocorrencia["tipo"] == "assassinato") &
    (ocorrencia["cidade"] == "João Pessoa")
]
print(caso["descricao"])

#obs: tive que usar a IA para entender e fazer, mas entendi que o formato usando "&" eh tipo uma estrutura condicional, que ele vai "percorrendo" todas as linhas e comparando se aquela linha é True em todas as condicoes; atribuindo à variável quando encontra



1227    As imagens de segurança mostram que houve 2 testemunhas. A primeira testemunha mora na última casa da "Avenida Ministro José Américo de Almeida". A segunda testemunha, chamada Maria Aparecida, mora em algum lugar da "Avenida Rui Carneiro".
Name: descricao, dtype: object


### Passo 2 — as testemunhas

In [ ]:
# TODO: 
pessoa = pd.read_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS)
pessoa.head()

moradores_avenida = pessoa[pessoa["rua"] == "Avenida Ministro José Américo de Almeida"] # -> cria um dicionario com todo mundo que mora la (chave) e o número da casa na rua (valor)

testemunha_1 = moradores_avenida.sort_values("numero_endereco", ascending=False).head(1) # ascending=False eh pra ordenar em ordem decrescente; head(1) pega o primeiro item
print(testemunha_1) # josenildo

testemunha_2 = pessoa[(pessoa["nome"].str.contains("Maria Aparecida", case=False, na=False)) & (pessoa["rua"] == "Avenida Rui Carneiro")] # str.contais eh pra buscar aquele nome, case=False ignora maiusculas e minusculas, na=False evita erros se tiver vazio, & segue a mesma estrutura condicional que falei na ultima celula
print(testemunha_2) # maria aparecida


        id                        nome  detran_id  numero_endereco  \
499  14887  Josenildo Pereira da Rocha     118009             4919   

                                          rua        cpf  
499  Avenida Ministro José Américo de Almeida  111564949  
        id                   nome  detran_id  numero_endereco  \
665  16371  Maria Aparecida Nunes     490173              103   

                      rua        cpf  
665  Avenida Rui Carneiro  318771143  


### Passo 3 — duas pistas, duas tabelas


In [ ]:
# TODO:
depoimento = pd.read_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS)
id_testemunha_1 = testemunha_1["id"].iloc[0] # dica do chat para extrair apenas o valor do id, pois sem isso, ele retorna um número a mais que corresponde à coluna 'Series' do pandas (nao entendi mas faz sentido, pois realmente tava aparecendo 2 números sem isso e ficava confuso entender qual era op ID)
id_testemunha_1

depoimento_testemunha_1 = depoimento[depoimento["pessoa_id"] == id_testemunha_1]
print(depoimento_testemunha_1["relato"].iloc[0])

# pelo depoimento de Josenildo do jose americo (deve ser vizinho de henrique), o MARGINAL tem plano ouro na academia kongo, o numero da matricula começa com 482 e entrou num carro que tem H42W na placa 

Eu ouvi um tiro e depois vi um homem saindo correndo. Ele tinha uma bolsa da "Academia Kongo". O número da matrícula na bolsa começava com "48Z". Só sócios do plano ouro têm essas bolsas. O homem entrou num carro com uma placa que continha "H42W".


In [ ]:
# TODO:
id_testemunha_2 = testemunha_2["id"].iloc[0]
id_testemunha_2

depoimento_testemunha_2 = depoimento[depoimento["pessoa_id"] == id_testemunha_2]
print(depoimento_testemunha_2["relato"].iloc[0])

# nossa dignissima Maria Aparecida após ver muitos videos de IA no reels lembrou que viu o assassinato acontecer e disse que o assassino treinou na mesma academia que ela no dia 9 de janeiro (tomara que a dona maria nao esteja se confundindo)


Eu vi o assassinato acontecer, e reconheci o assassino da minha academia, de quando eu estava treinando na semana passada, no dia 9 de janeiro.


### Passo 4 — cruzando as pistas

In [43]:
# TODO
membro_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS)
detran = pd.read_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS)

suspeitos_academia = membro_academia[(membro_academia["id"].str.startswith("48Z", na=False)) & (membro_academia["plano"].str.lower() == "ouro")]
# achar os suspeitos que tem plano ouro e n° da matrícula começando com 482 -> a sinteaxe do codigo tive que pesquisar como fazia
suspeitos_academia # dois suspeitos, Seu Severino e Seu Genildo (se tivessem vendo reels nao iam fazer isso)

suspeitos_veiculo = detran[detran["placa"].str.contains("H42W", case=False, na=False)]
suspeitos_veiculo # três suspeitos, com informacoes (menos o nome) deles 
 # um uno, um renegade e um fusca -> @PAX ajuda aqui


ids_pessoas_academia = suspeitos_academia["pessoa_id"]
ids_pessoas_academia

pessoas_suspeitas = pessoa[pessoa["id"].isin(ids_pessoas_academia)]
pessoas_suspeitas

ids_veiculos = suspeitos_veiculo["id"]

suspeitos = pessoas_suspeitas[pessoas_suspeitas["detran_id"].isin(ids_veiculos)]

veiculo_suspeito = suspeitos_veiculo[suspeitos_veiculo["id"].isin(suspeitos["detran_id"])]

print(suspeitos)
print(veiculo_suspeito)





         id                       nome  detran_id  numero_endereco  \
6327  67318  Genildo Cavalcanti Farias     423327              530   

                        rua        cpf  
6327  Washington Pl, Apt 3A  871539279  
          id  idade  altura cor_olhos cor_cabelo     genero   placa  \
3529  423327     30      70  castanho   castanho  masculino  0H42W2   

     marca_veiculo modelo_veiculo  
3529          Jeep       Renegade  


### Passo 5 — confirmar com o check-in

In [45]:
# TODO
checkin_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS)

matricula_suspeito = suspeitos_academia[suspeitos_academia["pessoa_id"].isin(suspeitos["id"])]
checkin_dia = checkin_academia[(checkin_academia["matricula_id"].isin(matricula_suspeito["id"])) & (checkin_academia["data_checkin"] == 20180109)]

print(matricula_suspeito)
print(checkin_dia)

        id  pessoa_id                       nome  data_matricula plano
182  48Z55      67318  Genildo Cavalcanti Farias        20160101  ouro
     matricula_id  data_checkin  hora_entrada  hora_saida
2701        48Z55      20180109          1530        1700


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `detran`) que fechou o caso |
| `pista_academia` | qual detalhe da matrícula (início do `id` + status do plano) bateu com o depoimento |
| `pista_veiculo` | qual trecho da placa bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [47]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver

df_resposta = pd.DataFrame([
    {
        "nome_suspeito": suspeitos["nome"].iloc[0],
        "placa_veiculo": veiculo_suspeito["placa"].iloc[0],
        "pista_academia": (
            f"matrícula {matricula_suspeito['id'].iloc[0]} iniciada por 48Z e plano {matricula_suspeito['plano'].iloc[0]}"
        ),
        "pista_veiculo": "a placa contém H42W",
        "justificativa": (
            "o meliante corresponde simultaneamente às pistas da matrícula da academia e da placa do veículo e o check-in em 09/01/2018 (às 15:30) confirma sua presença na academia na data dita pela testemunha"
        ),
    }
])

display(df_resposta)

df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)

resposta_publicada = pd.read_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS)

display(resposta_publicada)

,nome_suspeito,placa_veiculo,pista_academia,pista_veiculo,justificativa
0,Genildo Cavalcanti Farias,0H42W2,matrícula 48Z55 iniciada por 48Z e plano ouro,a placa contém H42W,o meliante corresponde simultaneamente às pistas da matrícula da academia e da placa do veículo e o check-in em 09/01/2018 (às 15:30) confirma sua presença na academia na data dita pela testemunha


,nome_suspeito,placa_veiculo,pista_academia,pista_veiculo,justificativa
0,Genildo Cavalcanti Farias,0H42W2,matrícula 48Z55 iniciada por 48Z e plano ouro,a placa contém H42W,o meliante corresponde simultaneamente às pistas da matrícula da academia e da placa do veículo e o check-in em 09/01/2018 (às 15:30) confirma sua presença na academia na data dita pela testemunha


---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.